<a id="top"></a>
# Time Domain Analysis: Photometry of Simulated Transients

***

## Kernel Information and Read-Only Status

To run this notebook, please select "Roman Research Nexus {VERSION}" kernel at the top right of your window. For example "Roman Research Nexus 2026.2".

This notebook is read-only. You can run cells and make edits, but you must save changes to a different location. We recommend saving the notebook within your home directory, or to a new folder within your home (e.g. <span style="font-variant:small-caps;">file > save notebook as > my-nbs/nb.ipynb</span>). Note that a directory must exist before you attempt to add a notebook to it.

## Introduction

This notebook continues the [time-domain simulation notebook](../time_domain_simulations/time_domain_simulations.ipynb) (from hereon the simulation notebook), which injects a Type Ia supernova into Roman WFI images and writes a time series of ASDF cutouts named `roman_sn1a_*_<filter>.asdf`.

Here we measure the same injected transient in those cutouts. The simulation notebook places the supernova at detector position $(X, Y) = (2000, 2000)$ and creates 400 pixel cutouts centered near $(X, Y) = (2050, 2050)$. In the cutouts analyzed here, the transient is therefore expected near $(x, y) = (150, 150)$.

We will:

- load the simulated cutout time series,
- perform background-subtracted aperture photometry,
- perform PSF-fitting photometry using an STPSF WFI PSF grid,
- compare the aperture and PSF measurements,
- estimate signal-to-noise ratios,
- compare recovered photometry against the injected `sncosmo` truth model,
- make a simple difference-image light curve,
- diagnose residual host-galaxy and background contamination,
- fit a normalized light-curve amplitude and peak time,
- repeat the measurement automatically for any additional simulated filters, and
- make a fun gif of the light curve corresponding to the cutouts.

This notebook measures instrumental image fluxes. If calibrated photometric metadata are available in later simulation products, the same workflow can be extended to convert the measured fluxes into physical units or AB magnitudes.

## Reference Data

This notebook uses `stpsf` to generate the WFI point-spread-function model used for PSF photometry and aperture-correction estimates.

### Local Run Settings

If you want to run the notebook on your local machine, refer to the [local installation](../../markdown/local-run.md) instructions before proceeding. Depending on which reference data are missing, the following cell may take several minutes to execute.

### On the Roman Research Nexus

If you are working on the Nexus, then the ancillary reference data are pre-installed and this cell will execute instantly.

In [ ]:
import os
import sys
import importlib.util

try:
    import notebook_data_dependencies as ndd
    local = True
except ImportError:
    local = False

# If running locally Get the directory with the script
if not local:
    notebook_dir = os.getcwd()
    shared_path = os.path.abspath(
        os.path.join(notebook_dir, '..', '..', 'shared', 'notebook_data_dependencies.py')
    )

    if os.path.exists(shared_path):
        print(f"Loading notebook_data_dependencies from shared location: {shared_path}")
        spec = importlib.util.spec_from_file_location("notebook_data_dependencies", shared_path)
        ndd = importlib.util.module_from_spec(spec)
        sys.modules['notebook_data_dependencies'] = ndd  # Optional: makes subsequent imports work
        spec.loader.exec_module(ndd)
    else:
        raise FileNotFoundError(f"Local install script not found at {shared_path}")

if not local:
    print("Running local data dependency installation...")
    result = ndd.install_files(packages=['stpsf'])

    # Update environment variables (if necessary) and print reference data paths
    print('Reference data paths set to:')
    for k, v in result.items():
        if not v['pre_installed']:
            os.environ[k] = v['path']
        print(f"\t{k} = {v['path']}")

else:
    print("Running on RNN — data already available, skipping local install.")

## Imports

We use the standard NumPy and Astropy stack, plus packages used elsewhere in the Roman notebook collection:

- *asdf* for reading Roman ASDF cutouts
- *astropy.table* for storing photometry results
- *astropy.stats* for sigma clipping background pixels
- *astropy.visualization* for image normalization
- *IPython.display* for rendering generated GIF animations in the notebook
- *matplotlib* for plotting images, light curves, and animations
- *numpy* for array operations
- *photutils* for aperture definitions and aperture statistics
- *Pillow* for reading GIF frames from the simulation notebook animation
- *sncosmo* for recreating and fitting the simulated supernova model
- *stpsf* for Roman WFI PSF models

In [ ]:
from pathlib import Path
import re
import warnings

import asdf
import astropy.units as u
from astropy.stats import SigmaClip
from astropy.table import Table, vstack
from astropy.time import Time
from astropy.visualization import simple_norm
from IPython.display import Image as NotebookImage, display
from matplotlib import animation
from matplotlib.colors import TwoSlopeNorm
import matplotlib.pyplot as plt
import numpy as np
from photutils.aperture import CircularAnnulus, CircularAperture, ApertureStats
from PIL import Image as PILImage, ImageSequence
import sncosmo
import stpsf

warnings.filterwarnings('ignore', module='astropy.wcs')

## Tutorial Data

This analysis starts from the cutout ASDF files created near the end of the simulation notebook. By default we look for all simulated filters:

`roman_sn1a_*_*.asdf`

The detailed walkthrough uses `F129` as the primary filter when available, matching the default simulation notebook path. If you later uncomment the F184 simulation cells, or add additional filters, this notebook will discover those files and process them in the multi-filter section below.

Each filename contains the integer Modified Julian Date (MJD) for that simulated epoch. The helper functions below sort the files by filter and MJD and load the `roman/data` array from each ASDF file.

Running the time-domain simulation notebook first is the recommended path because it creates Roman I-Sim products with the intended host galaxy and instrument modeling. If those files are not present, this notebook can create a small synthetic demo data set with the same filename and ASDF layout so the analysis workflow remains runnable end-to-end.

In [ ]:
data_dir = Path('.')
cutout_pattern = 'roman_sn1a_*_*.asdf'
preferred_filter = 'F129'
allow_demo_data = True
demo_data_dir = data_dir / 'demo_cutouts'


def extract_mjd(path):
    """Extract the integer MJD encoded in a simulation cutout filename."""
    match = re.search(r'roman_sn1a_(\d+)_([a-z0-9]+)\.asdf$', path.name.lower())
    if match is None:
        raise ValueError(f'Could not parse MJD and filter from {path.name}')
    return float(match.group(1))


def extract_filter(path):
    """Extract the WFI filter name encoded in a simulation cutout filename."""
    match = re.search(r'roman_sn1a_(\d+)_([a-z0-9]+)\.asdf$', path.name.lower())
    if match is None:
        raise ValueError(f'Could not parse MJD and filter from {path.name}')
    return match.group(2).upper()


def discover_cutout_files(directory):
    """Find all cutout files in a directory and sort them by filter and MJD."""
    return sorted(
        directory.glob(cutout_pattern),
        key=lambda path: (extract_filter(path), extract_mjd(path)),
    )


def demo_light_curve_maggies(mjds, filter_name):
    """Return an injected demo light curve in maggies."""
    try:
        demo_model = sncosmo.Model(source='salt3-nir')
        demo_t0 = Time('2027-06-08T13:25:42').mjd
        demo_model.set(z=1.2, t0=demo_t0, x1=1, c=1)
        demo_model.set_source_peakmag(20, 'f158', 'ab')
        demo_abmag = demo_model.bandmag(filter_name.lower(), 'ab', mjds)
        return 10 ** (-0.4 * demo_abmag)
    except Exception:
        demo_t0 = Time('2027-06-08T13:25:42').mjd
        phase = np.asarray(mjds, dtype=float) - demo_t0
        return 10 ** (-0.4 * 20.0) * np.exp(-0.5 * (phase / 22.0) ** 2)


def create_demo_cutouts(output_dir, filter_name='F129'):
    """Create lightweight ASDF cutouts when Roman I-Sim products are unavailable."""
    output_dir.mkdir(exist_ok=True)
    demo_files = discover_cutout_files(output_dir)
    if len(demo_files) > 0:
        return demo_files

    rng = np.random.default_rng(202602)
    demo_t0 = Time('2027-06-08T13:25:42').mjd
    demo_mjds = np.arange(-40, 115, 5, dtype=float) + demo_t0
    demo_mjds = np.round(demo_mjds).astype(int)
    demo_shape = (400, 400)
    y_grid, x_grid = np.indices(demo_shape, dtype=float)
    source_x, source_y = 150.0, 150.0

    psf_sigma = 1.25
    psf_image = np.exp(-0.5 * (((x_grid - source_x) / psf_sigma) ** 2 + ((y_grid - source_y) / psf_sigma) ** 2))
    psf_image /= np.sum(psf_image)

    host_x, host_y = 154.0, 147.0
    host_radius = np.sqrt(((x_grid - host_x) / 18.0) ** 2 + ((y_grid - host_y) / 11.0) ** 2)
    host_image = 0.18 + 2.4 * np.exp(-host_radius)
    host_image += 0.03 * (x_grid - np.mean(x_grid)) / np.max(x_grid)

    demo_maggies = demo_light_curve_maggies(demo_mjds, filter_name)
    finite = np.isfinite(demo_maggies)
    if not np.any(finite) or np.nanmax(demo_maggies) <= 0:
        demo_maggies = np.exp(-0.5 * ((demo_mjds - demo_t0) / 22.0) ** 2)
    normalized_flux = demo_maggies / np.nanmax(demo_maggies)

    for mjd, relative_flux in zip(demo_mjds, normalized_flux):
        transient_flux = 460.0 * relative_flux
        noise = rng.normal(0.0, 0.035, demo_shape)
        image = host_image + transient_flux * psf_image + noise
        output_path = output_dir / f'roman_sn1a_{mjd}_{filter_name.lower()}.asdf'
        asdf.AsdfFile({'roman': {'data': image.astype(np.float32)}}).write_to(output_path)

    return discover_cutout_files(output_dir)


all_cutout_files = discover_cutout_files(data_dir)
cutout_data_origin = 'Roman I-Sim simulation outputs'

if len(all_cutout_files) == 0:
    if allow_demo_data:
        print(
            f'No files matching {cutout_pattern!r} were found in {data_dir.resolve()}. '
            'Creating lightweight demo cutouts instead.'
        )
        data_dir = demo_data_dir
        all_cutout_files = create_demo_cutouts(data_dir, preferred_filter)
        cutout_data_origin = 'lightweight demo cutouts'
    else:
        raise FileNotFoundError(
            f'No files matching {cutout_pattern!r} were found in {data_dir.resolve()}. '
            'Run the time-domain simulation notebook first, set allow_demo_data=True, or update cutout_pattern.'
        )

available_filters = sorted({extract_filter(path) for path in all_cutout_files})
primary_filter = preferred_filter if preferred_filter in available_filters else available_filters[0]
cutout_files = [
    path for path in all_cutout_files
    if extract_filter(path) == primary_filter
]

print(f'Data origin: {cutout_data_origin}')
print(f'Found {len(all_cutout_files)} total cutout files across filters: {available_filters}')
print(f'Using {primary_filter} for the detailed single-filter walkthrough.')
cutout_files[:5]

In [ ]:
def read_cutout(path):
    """Read the Roman data array from one ASDF cutout file."""
    with asdf.open(path) as af:
        image = np.array(af['roman']['data'], dtype=float)
    return image


images = np.array([read_cutout(path) for path in cutout_files])
mjds = np.array([extract_mjd(path) for path in cutout_files])
filters = np.array([extract_filter(path) for path in cutout_files])

def estimate_source_position(image, initial_position=(150.0, 150.0), half_size=10):
    """Estimate the source centroid in a small box around an initial position."""
    x0, y0 = initial_position
    x_min = max(int(np.floor(x0)) - half_size, 0)
    x_max = min(int(np.floor(x0)) + half_size + 1, image.shape[1])
    y_min = max(int(np.floor(y0)) - half_size, 0)
    y_max = min(int(np.floor(y0)) + half_size + 1, image.shape[0])

    stamp = image[y_min:y_max, x_min:x_max]
    yy, xx = np.mgrid[y_min:y_max, x_min:x_max]
    weights = stamp - np.nanmedian(stamp)
    weights = np.clip(weights, 0, None)

    if not np.any(np.isfinite(weights)) or np.nansum(weights) <= 0:
        return initial_position

    x_centroid = np.nansum(xx * weights) / np.nansum(weights)
    y_centroid = np.nansum(yy * weights) / np.nansum(weights)
    return (float(x_centroid), float(y_centroid))


peak_index = np.nanargmax([
    np.nanmax(image[140:161, 140:161])
    for image in images
])

# Geometry inherited from the simulation notebook, refined from the peak epoch.
nominal_cutout_target_position = (150.0, 150.0)
nominal_full_detector_target_position = (2000.0, 2000.0)
cutout_target_position = estimate_source_position(images[peak_index], nominal_cutout_target_position)
detector_to_cutout_offset = (
    nominal_full_detector_target_position[0] - nominal_cutout_target_position[0],
    nominal_full_detector_target_position[1] - nominal_cutout_target_position[1],
)
full_detector_target_position = (
    cutout_target_position[0] + detector_to_cutout_offset[0],
    cutout_target_position[1] + detector_to_cutout_offset[1],
)
filter_name = primary_filter
detector_name = 'SCA01'

print(f'Nominal cutout source position: {nominal_cutout_target_position}')
print(f'Refined cutout source position: ({cutout_target_position[0]:.2f}, {cutout_target_position[1]:.2f})')

Table({
    'file': [path.name for path in cutout_files[:5]],
    'mjd': mjds[:5],
    'filter': filters[:5],
})

## Display the Simulated Transient

Before measuring the light curve, we inspect one epoch near peak brightness. The red circle marks the fixed forced-photometry position inherited from the simulation notebook.

In [ ]:
peak_index = np.nanargmax([
    np.nanmax(image[140:161, 140:161])
    for image in images
])

fig, ax = plt.subplots(figsize=(6, 6))
norm = simple_norm(images[peak_index], 'asinh', percent=99.5)
ax.imshow(images[peak_index], origin='lower', cmap='afmhot', norm=norm)

aperture = CircularAperture(cutout_target_position, r=3)
aperture.plot(ax=ax, color='red', lw=2)

ax.set_title(f'{filters[peak_index]} simulated epoch: MJD {mjds[peak_index]:.0f}')
ax.set_xlabel('Cutout X (pixels)')
ax.set_ylabel('Cutout Y (pixels)')

## Aperture Photometry

Forced aperture photometry uses a predefined source position and aperture size for every epoch. This is a natural first pass for a simulated time series because the source position is known from the injection step. For more on aperture photometry see the [Aperture Photometry](../aperture_photometry/aperture_photometry.ipynb) tutorial.

We estimate a local background from a circular annulus around the source and subtract that background from the circular source aperture. We also compute an optional aperture correction from the STPSF encircled-energy profile for the WFI detector, filter, and source position.

In [ ]:
aperture_radius = 3.0  # pixels
annulus_inner_radius = 8.0  # pixels
annulus_outer_radius = 15.0  # pixels
pixel_scale = 0.11 * u.arcsec / u.pixel


def aperture_correction_from_stpsf(filter_name, detector_name, detector_position, radius_pix):
    """Estimate the aperture correction from an STPSF encircled-energy curve."""
    wfi = stpsf.roman.WFI()
    wfi.filter = filter_name.upper()
    wfi.detector = detector_name
    wfi.detector_position = detector_position

    psf = wfi.calc_psf(display=False, nlambda=1)
    ee_func = stpsf.measure_ee(psf)
    radius_arcsec = (radius_pix * u.pixel * pixel_scale).to_value(u.arcsec)
    return 1.0 / ee_func(radius_arcsec)


aperture_correction_cache = {}
aperture_correction = aperture_correction_from_stpsf(
    filter_name=filter_name,
    detector_name=detector_name,
    detector_position=full_detector_target_position,
    radius_pix=aperture_radius,
)
aperture_correction_cache[filter_name] = aperture_correction

print(f'{filter_name} aperture correction for r = {aperture_radius:.1f} pixels: {aperture_correction:.3f}')

In [ ]:
def measure_aperture_flux(image, position, radius, inner_radius, outer_radius, correction=1.0):
    """Measure local-background-subtracted aperture flux for one image."""
    source_aperture = CircularAperture(position, r=radius)
    sky_annulus = CircularAnnulus(position, r_in=inner_radius, r_out=outer_radius)

    source_stats = ApertureStats(image, source_aperture)
    sky_stats = ApertureStats(
        image,
        sky_annulus,
        sigma_clip=SigmaClip(sigma=3.0, maxiters=5),
    )

    aperture_area = source_aperture.area
    background_per_pixel = sky_stats.median
    background_total = background_per_pixel * aperture_area
    raw_flux = source_stats.sum
    net_flux = raw_flux - background_total

    # A compact uncertainty estimate from the scatter in the local sky annulus.
    flux_err = np.sqrt(aperture_area) * sky_stats.std

    return {
        'raw_flux': raw_flux,
        'background_per_pixel': background_per_pixel,
        'background_total': background_total,
        'aperture_flux': net_flux * correction,
        'aperture_flux_err': flux_err * correction,
    }


aperture_rows = []
for path, mjd, filt, image in zip(cutout_files, mjds, filters, images):
    result = measure_aperture_flux(
        image,
        cutout_target_position,
        aperture_radius,
        annulus_inner_radius,
        annulus_outer_radius,
        correction=aperture_correction,
    )
    aperture_rows.append({
        'file': path.name,
        'mjd': mjd,
        'filter': filt,
        **result,
    })

aperture_table = Table(rows=aperture_rows)
aperture_table[:5]

## PSF Photometry

PSF photometry estimates the source flux by fitting a point-spread-function model directly to the pixels near the source. This is often useful when sources are faint, crowded, or when an aperture includes significant host-galaxy light.

The [STPSF tutorial](../stpsf/stpsf.ipynb) shows how to generate a grid of Roman WFI PSFs and evaluate it at arbitrary detector positions. We use that same pattern here. The PSF grid is evaluated at the refined source position, and each epoch is fit with a point source plus a local background plane:

$$
I(x, y) = F_{\rm PSF} P(x, y) + B_0 + B_x (x - x_0) + B_y (y - y_0)
$$

where $P(x, y)$ is a unit-normalized PSF image, $F_{\rm PSF}$ is the fitted source flux, and the background-plane terms capture smooth host-galaxy light across the small fitting stamp.

In [ ]:
psf_fit_half_size = 15  # pixels


def make_psf_grid(filter_name, detector_name, detector_position):
    """Create an STPSF grid for one WFI filter and detector position."""
    wfi = stpsf.roman.WFI()
    wfi.filter = filter_name.upper()
    wfi.detector = detector_name
    wfi.detector_position = detector_position
    return wfi.psf_grid(num_psfs=9, all_detectors=False)


# A small PSF grid is enough for this single-detector, single-position example.
psf_grid_cache = {}
psf_grid = make_psf_grid(filter_name, detector_name, full_detector_target_position)
psf_grid_cache[filter_name] = psf_grid

In [ ]:
def measure_psf_flux(
    image,
    cutout_position,
    full_detector_position,
    psf_grid,
    half_size=15,
    background_model='plane',
):
    """Fit a unit-normalized STPSF model plus a local background model."""
    x0, y0 = cutout_position
    x_min = int(np.round(x0)) - half_size
    x_max = int(np.round(x0)) + half_size + 1
    y_min = int(np.round(y0)) - half_size
    y_max = int(np.round(y0)) + half_size + 1

    stamp = image[y_min:y_max, x_min:x_max]
    yy_cutout, xx_cutout = np.mgrid[y_min:y_max, x_min:x_max]

    x_offset = full_detector_position[0] - cutout_position[0]
    y_offset = full_detector_position[1] - cutout_position[1]
    xx_detector = xx_cutout + x_offset
    yy_detector = yy_cutout + y_offset

    psf_unit = psf_grid.evaluate(
        x=xx_detector,
        y=yy_detector,
        flux=1.0,
        x_0=full_detector_position[0],
        y_0=full_detector_position[1],
    )
    psf_unit = np.array(psf_unit, dtype=float)
    psf_unit /= np.nansum(psf_unit)

    mask = np.isfinite(stamp) & np.isfinite(psf_unit)
    design_columns = [
        psf_unit[mask].ravel(),
        np.ones(np.count_nonzero(mask)),
    ]

    if background_model == 'plane':
        design_columns.extend([
            (xx_cutout[mask] - x0).ravel(),
            (yy_cutout[mask] - y0).ravel(),
        ])
    elif background_model != 'constant':
        raise ValueError("background_model must be 'constant' or 'plane'")

    design_matrix = np.column_stack(design_columns)
    data_vector = stamp[mask].ravel()

    fit_values, _, _, _ = np.linalg.lstsq(design_matrix, data_vector, rcond=None)
    flux = fit_values[0]
    background = fit_values[1]
    background_x_slope = fit_values[2] if background_model == 'plane' else 0.0
    background_y_slope = fit_values[3] if background_model == 'plane' else 0.0

    background_image = background + background_x_slope * (xx_cutout - x0) + background_y_slope * (yy_cutout - y0)
    model = flux * psf_unit + background_image
    residual = stamp - model

    dof = max(np.count_nonzero(mask) - design_matrix.shape[1], 1)
    residual_variance = np.nansum(residual[mask] ** 2) / dof
    covariance = residual_variance * np.linalg.pinv(design_matrix.T @ design_matrix)
    flux_err = np.sqrt(covariance[0, 0])

    return {
        'psf_flux': flux,
        'psf_flux_err': flux_err,
        'psf_background': background,
        'psf_background_x_slope': background_x_slope,
        'psf_background_y_slope': background_y_slope,
        'psf_x': x0,
        'psf_y': y0,
        'psf_residual_rms': np.sqrt(np.nanmean(residual[mask] ** 2)),
        'psf_max_abs_residual_fraction': np.nanmax(np.abs(residual)) / np.nanmax(np.abs(stamp)),
        'stamp': stamp,
        'psf_model': model,
        'psf_background_image': background_image,
        'psf_residual': residual,
    }


psf_rows = []
psf_diagnostics = {}
for path, mjd, filt, image in zip(cutout_files, mjds, filters, images):
    result = measure_psf_flux(
        image,
        cutout_target_position,
        full_detector_target_position,
        psf_grid,
        half_size=psf_fit_half_size,
    )
    psf_rows.append({
        'file': path.name,
        'mjd': mjd,
        'filter': filt,
        'psf_flux': result['psf_flux'],
        'psf_flux_err': result['psf_flux_err'],
        'psf_background': result['psf_background'],
        'psf_background_x_slope': result['psf_background_x_slope'],
        'psf_background_y_slope': result['psf_background_y_slope'],
        'psf_x': result['psf_x'],
        'psf_y': result['psf_y'],
        'psf_residual_rms': result['psf_residual_rms'],
        'psf_max_abs_residual_fraction': result['psf_max_abs_residual_fraction'],
    })
    psf_diagnostics[path.name] = result

psf_table = Table(rows=psf_rows)
psf_table[:5]

### Inspect the PSF Fit

The panels below show the data, fitted PSF model, and residual image for the epoch near peak brightness. The data and model panels use the same intensity stretch; the residual panel uses a zero-centered diverging stretch so positive and negative structure are easier to interpret. These simulated cutouts do not carry a reliable physical data unit in the ASDF metadata, so the color bars are labeled in simulated L2 image units.

In [ ]:
peak_file = cutout_files[peak_index].name
diagnostic = psf_diagnostics[peak_file]

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(12, 4), constrained_layout=True)

titles = ['Data', 'PSF model + background', 'Residual']
arrays = [
    diagnostic['stamp'],
    diagnostic['psf_model'],
    diagnostic['psf_residual'],
]

data_model_values = np.concatenate([
    diagnostic['stamp'].ravel(),
    diagnostic['psf_model'].ravel(),
])
data_model_vmin, data_model_vmax = np.nanpercentile(data_model_values, [1, 99.5])
residual_limit = np.nanpercentile(np.abs(diagnostic['psf_residual']), 99)
image_unit_label = 'Simulated L2 image units'

data_model_images = []
for ax, title, array in zip(axes[:2], titles[:2], arrays[:2]):
    image = ax.imshow(
        array,
        origin='lower',
        cmap='afmhot',
        vmin=data_model_vmin,
        vmax=data_model_vmax,
    )
    data_model_images.append(image)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

residual_image = axes[2].imshow(
    diagnostic['psf_residual'],
    origin='lower',
    cmap='RdBu_r',
    norm=TwoSlopeNorm(vcenter=0, vmin=-residual_limit, vmax=residual_limit),
)
axes[2].set_title(titles[2])
axes[2].set_xticks([])
axes[2].set_yticks([])

data_model_colorbar = fig.colorbar(
    data_model_images[0],
    ax=[axes[0], axes[1]],
    shrink=0.82,
    pad=0.02,
)
data_model_colorbar.set_label(image_unit_label)

residual_colorbar = fig.colorbar(
    residual_image,
    ax=axes[2],
    shrink=0.82,
    pad=0.02,
)
residual_colorbar.set_label(f'Data - model ({image_unit_label})')

fig.suptitle(f'PSF fit diagnostics: {peak_file}')

print(f"Fitted source position: x={diagnostic['psf_x']:.2f}, y={diagnostic['psf_y']:.2f}")
print(f"PSF flux: {diagnostic['psf_flux']:.2f} +/- {diagnostic['psf_flux_err']:.2f}")
print(f"Residual RMS: {diagnostic['psf_residual_rms']:.3f}")
print(
    'Max absolute residual / max data value: '
    f"{diagnostic['psf_max_abs_residual_fraction']:.3f}"
)

## Compare Aperture and PSF Photometry

Now we combine the aperture and PSF measurements into one table. The two methods will not be identical because they treat the host-galaxy light differently: aperture photometry subtracts a local annulus background, while PSF photometry fits a compact point-source component plus a constant local background across the fitting stamp.

In [ ]:
def safe_ratio(numerator, denominator):
    """Return numerator / denominator while avoiding divide-by-zero warnings."""
    numerator = np.asarray(numerator, dtype=float)
    denominator = np.asarray(denominator, dtype=float)
    return np.divide(
        numerator,
        denominator,
        out=np.full_like(numerator, np.nan, dtype=float),
        where=np.isfinite(denominator) & (denominator != 0),
    )


photometry = aperture_table.copy()
photometry['psf_flux'] = psf_table['psf_flux']
photometry['psf_flux_err'] = psf_table['psf_flux_err']
photometry['psf_background'] = psf_table['psf_background']
photometry['psf_background_x_slope'] = psf_table['psf_background_x_slope']
photometry['psf_background_y_slope'] = psf_table['psf_background_y_slope']
photometry['psf_x'] = psf_table['psf_x']
photometry['psf_y'] = psf_table['psf_y']
photometry['psf_residual_rms'] = psf_table['psf_residual_rms']
photometry['psf_max_abs_residual_fraction'] = psf_table['psf_max_abs_residual_fraction']

photometry['flux_difference'] = photometry['aperture_flux'] - photometry['psf_flux']
photometry['flux_ratio'] = safe_ratio(photometry['aperture_flux'], photometry['psf_flux'])
photometry['aperture_snr'] = safe_ratio(photometry['aperture_flux'], photometry['aperture_flux_err'])
photometry['psf_snr'] = safe_ratio(photometry['psf_flux'], photometry['psf_flux_err'])
photometry['aperture_background_fraction'] = safe_ratio(
    photometry['background_total'],
    photometry['raw_flux'],
)

for column in ('aperture_flux', 'psf_flux'):
    peak_flux = np.nanmax(photometry[column])
    photometry[f'{column}_normalized'] = photometry[column] / peak_flux

photometry[:5]

### Signal-to-Noise Ratio

The aperture and PSF flux uncertainties are estimated differently, so their signal-to-noise ratios (S/N) should be interpreted as compact diagnostics rather than final uncertainties. They are still useful for checking whether the two extraction methods identify the same high-quality epochs.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(photometry['mjd'], photometry['aperture_snr'], marker='o', label='Aperture S/N')
ax.plot(photometry['mjd'], photometry['psf_snr'], marker='s', label='PSF S/N')

ax.axhline(5, color='0.4', linestyle='--', linewidth=1, label='S/N = 5')
ax.set_xlabel('MJD')
ax.set_ylabel('Signal-to-noise ratio')
ax.set_title(f'{filter_name} photometric signal-to-noise')
ax.grid(alpha=0.3)
ax.legend()

In [ ]:
fig, (ax_lc, ax_ratio) = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(8, 7),
    sharex=True,
    gridspec_kw={'height_ratios': [3, 1]},
    constrained_layout=True,
)

ax_lc.errorbar(
    photometry['mjd'],
    photometry['aperture_flux'],
    yerr=photometry['aperture_flux_err'],
    fmt='o-',
    label='Aperture photometry',
)
ax_lc.errorbar(
    photometry['mjd'],
    photometry['psf_flux'],
    yerr=photometry['psf_flux_err'],
    fmt='s-',
    label='PSF photometry',
)
ax_lc.set_ylabel('Flux (image units)')
ax_lc.set_title(f'{filter_name} light curve of the simulated supernova')
ax_lc.legend()

ax_ratio.scatter(photometry['mjd'], photometry['flux_ratio'])
ax_ratio.axhline(1.0, color='0.4', linestyle='--', linewidth=1)
ax_ratio.set_xlabel('MJD')
ax_ratio.set_ylabel('Aper. / PSF')

for ax in (ax_lc, ax_ratio):
    ax.grid(alpha=0.3)

It is also useful to compare the two light-curve shapes after normalizing each method by its own peak flux. This emphasizes differences in the recovered time evolution rather than the absolute flux scale.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    photometry['mjd'],
    photometry['aperture_flux_normalized'],
    marker='o',
    label='Aperture photometry',
)
ax.plot(
    photometry['mjd'],
    photometry['psf_flux_normalized'],
    marker='s',
    label='PSF photometry',
)

ax.set_xlabel('MJD')
ax.set_ylabel('Flux / peak flux')
ax.set_title(f'Normalized {filter_name} light-curve comparison')
ax.grid(alpha=0.3)
ax.legend()

## Light-Curve Analysis with sncosmo

The simulation notebook generated the injected transient with `sncosmo` using the SALT3-NIR Type Ia supernova model. We can recreate that same model here and compare its expected light-curve shape with the recovered aperture and PSF photometry.

The measured image fluxes are not in `sncosmo` flux units, so we fit a simple linear transformation between the model light curve and each measured light curve:

$$
f_{\rm measured}(t) = A f_{\rm sncosmo}(t) + B
$$

where $A$ is the flux scale and $B$ is a constant residual background term. This is enough to compare the recovered time evolution and estimate whether aperture and PSF photometry peak at the expected epoch.

In [ ]:
sncosmo_model = sncosmo.Model(source='salt3-nir')

simulation_t0 = Time('2027-06-08T13:25:42').mjd
sncosmo_model.set(
    z=1.2,
    t0=simulation_t0,
    x1=1,
    c=1,
)
sncosmo_model.set_source_peakmag(20, 'f158', 'ab')


def sncosmo_maggies_for_filter(filter_name, mjd, model=sncosmo_model):
    """Evaluate the sncosmo model in one WFI filter and return maggies."""
    abmag = model.bandmag(filter_name.lower(), 'ab', mjd)
    return 10 ** (-0.4 * abmag)


sncosmo_band = filter_name.lower()
model_abmag = sncosmo_model.bandmag(sncosmo_band, 'ab', photometry['mjd'])
model_maggies = 10 ** (-0.4 * model_abmag)
model_maggies_normalized = model_maggies / np.nanmax(model_maggies)

photometry['sncosmo_model_abmag'] = model_abmag
photometry['sncosmo_model_maggies'] = model_maggies
photometry['sncosmo_model_normalized'] = model_maggies_normalized

dense_mjd = np.linspace(np.nanmin(mjds), np.nanmax(mjds), 400)
dense_abmag = sncosmo_model.bandmag(sncosmo_band, 'ab', dense_mjd)
dense_maggies = 10 ** (-0.4 * dense_abmag)
dense_maggies_normalized = dense_maggies / np.nanmax(dense_maggies)
model_peak_mjd = dense_mjd[np.nanargmax(dense_maggies)]

print(f'{filter_name} sncosmo model peak MJD: {model_peak_mjd:.2f}')

In [ ]:
def fit_scaled_sncosmo_model(measured_flux, measured_err, model_flux):
    """Fit measured_flux = scale * model_flux + offset."""
    measured_flux = np.asarray(measured_flux, dtype=float)
    measured_err = np.asarray(measured_err, dtype=float)
    model_flux = np.asarray(model_flux, dtype=float)

    mask = np.isfinite(measured_flux) & np.isfinite(model_flux)
    if np.any(np.isfinite(measured_err) & (measured_err > 0)):
        mask &= np.isfinite(measured_err) & (measured_err > 0)
        weights = 1.0 / measured_err[mask] ** 2
    else:
        weights = np.ones(np.count_nonzero(mask))

    design_matrix = np.column_stack([model_flux[mask], np.ones(np.count_nonzero(mask))])
    sqrt_weights = np.sqrt(weights)
    weighted_design = design_matrix * sqrt_weights[:, np.newaxis]
    weighted_flux = measured_flux[mask] * sqrt_weights

    scale, offset = np.linalg.lstsq(weighted_design, weighted_flux, rcond=None)[0]
    fitted_flux = scale * model_flux + offset
    residual = measured_flux - fitted_flux

    chi2 = np.sum(((measured_flux[mask] - fitted_flux[mask]) ** 2) * weights)
    dof = max(np.count_nonzero(mask) - 2, 1)

    return {
        'scale': scale,
        'offset': offset,
        'fitted_flux': fitted_flux,
        'residual': residual,
        'reduced_chi2': chi2 / dof,
    }


aperture_sncosmo_fit = fit_scaled_sncosmo_model(
    photometry['aperture_flux'],
    photometry['aperture_flux_err'],
    photometry['sncosmo_model_maggies'],
)
psf_sncosmo_fit = fit_scaled_sncosmo_model(
    photometry['psf_flux'],
    photometry['psf_flux_err'],
    photometry['sncosmo_model_maggies'],
)

photometry['aperture_sncosmo_fit_flux'] = aperture_sncosmo_fit['fitted_flux']
photometry['aperture_sncosmo_residual'] = aperture_sncosmo_fit['residual']
photometry['psf_sncosmo_fit_flux'] = psf_sncosmo_fit['fitted_flux']
photometry['psf_sncosmo_residual'] = psf_sncosmo_fit['residual']

aperture_peak_mjd = photometry['mjd'][np.nanargmax(photometry['aperture_flux'])]
psf_peak_mjd = photometry['mjd'][np.nanargmax(photometry['psf_flux'])]

sncosmo_summary = Table(rows=[
    {
        'method': 'Aperture',
        'scale': aperture_sncosmo_fit['scale'],
        'offset': aperture_sncosmo_fit['offset'],
        'reduced_chi2': aperture_sncosmo_fit['reduced_chi2'],
        'measured_peak_mjd': aperture_peak_mjd,
        'model_peak_mjd': model_peak_mjd,
        'peak_offset_days': aperture_peak_mjd - model_peak_mjd,
    },
    {
        'method': 'PSF',
        'scale': psf_sncosmo_fit['scale'],
        'offset': psf_sncosmo_fit['offset'],
        'reduced_chi2': psf_sncosmo_fit['reduced_chi2'],
        'measured_peak_mjd': psf_peak_mjd,
        'model_peak_mjd': model_peak_mjd,
        'peak_offset_days': psf_peak_mjd - model_peak_mjd,
    },
])

sncosmo_summary

### Model-Calibrated Magnitudes

The aperture and PSF fluxes above are measured in image units. To express them as magnitudes, we use the fitted `sncosmo` scale and offset to convert each measured flux back into model maggies, then compute AB magnitudes. Vega magnitudes are also added when the local `sncosmo` Vega magnitude system is available.

In [ ]:
def maggies_to_abmag(maggies, maggies_err=None):
    """Convert linear AB maggies to AB magnitudes and optional magnitude errors."""
    maggies = np.asarray(maggies, dtype=float)
    abmag = np.full(maggies.shape, np.nan, dtype=float)
    positive = np.isfinite(maggies) & (maggies > 0)
    abmag[positive] = -2.5 * np.log10(maggies[positive])

    if maggies_err is None:
        return abmag, None

    maggies_err = np.asarray(maggies_err, dtype=float)
    abmag_err = np.full(maggies.shape, np.nan, dtype=float)
    err_mask = positive & np.isfinite(maggies_err) & (maggies_err >= 0)
    abmag_err[err_mask] = (2.5 / np.log(10)) * maggies_err[err_mask] / maggies[err_mask]
    return abmag, abmag_err


vega_offset_cache = {}


def get_ab_minus_vega_offset(filter_name, mjd, model=sncosmo_model):
    """Return AB - Vega for one filter, or NaN if Vega data are unavailable."""
    cache_key = filter_name.upper()
    if cache_key in vega_offset_cache:
        return vega_offset_cache[cache_key]

    try:
        abmag = model.bandmag(filter_name.lower(), 'ab', mjd)
        vegamag = model.bandmag(filter_name.lower(), 'vega', mjd)
        offsets = np.asarray(abmag, dtype=float) - np.asarray(vegamag, dtype=float)
        offset = np.nanmedian(offsets[np.isfinite(offsets)])
    except Exception as exc:
        print(
            f'Vega magnitudes are unavailable for {filter_name.upper()} '
            f'({exc.__class__.__name__}); Vega columns will be NaN.'
        )
        offset = np.nan

    vega_offset_cache[cache_key] = offset
    return offset


def ensure_float_column(table, column_name):
    """Create a float column filled with NaN if it is not already present."""
    if column_name not in table.colnames:
        table[column_name] = np.full(len(table), np.nan, dtype=float)


def add_model_calibrated_magnitude_columns(
    table,
    flux_column,
    flux_err_column,
    fit_result,
    prefix,
    ab_minus_vega=np.nan,
    mask=None,
):
    """Add model-calibrated maggies, AB magnitudes, and Vega magnitudes."""
    if mask is None:
        mask = np.ones(len(table), dtype=bool)

    columns = [
        f'{prefix}_maggies_calibrated',
        f'{prefix}_maggies_calibrated_err',
        f'{prefix}_abmag',
        f'{prefix}_abmag_err',
        f'{prefix}_vega_mag',
        f'{prefix}_vega_mag_err',
    ]
    for column in columns:
        ensure_float_column(table, column)

    scale = float(fit_result['scale'])
    offset = float(fit_result['offset'])
    if np.isfinite(scale) and scale != 0:
        measured_flux = np.asarray(table[flux_column][mask], dtype=float)
        measured_err = np.asarray(table[flux_err_column][mask], dtype=float)
        maggies = (measured_flux - offset) / scale
        maggies_err = measured_err / np.abs(scale)
    else:
        maggies = np.full(np.count_nonzero(mask), np.nan, dtype=float)
        maggies_err = np.full(np.count_nonzero(mask), np.nan, dtype=float)

    abmag, abmag_err = maggies_to_abmag(maggies, maggies_err)
    table[f'{prefix}_maggies_calibrated'][mask] = maggies
    table[f'{prefix}_maggies_calibrated_err'][mask] = maggies_err
    table[f'{prefix}_abmag'][mask] = abmag
    table[f'{prefix}_abmag_err'][mask] = abmag_err

    if np.isfinite(ab_minus_vega):
        table[f'{prefix}_vega_mag'][mask] = abmag - ab_minus_vega
    table[f'{prefix}_vega_mag_err'][mask] = abmag_err


ab_minus_vega = get_ab_minus_vega_offset(filter_name, photometry['mjd'])
if np.isfinite(ab_minus_vega):
    photometry['sncosmo_model_vega_mag'] = photometry['sncosmo_model_abmag'] - ab_minus_vega
else:
    photometry['sncosmo_model_vega_mag'] = np.full(len(photometry), np.nan, dtype=float)

add_model_calibrated_magnitude_columns(
    photometry,
    'aperture_flux',
    'aperture_flux_err',
    aperture_sncosmo_fit,
    'aperture',
    ab_minus_vega=ab_minus_vega,
)
add_model_calibrated_magnitude_columns(
    photometry,
    'psf_flux',
    'psf_flux_err',
    psf_sncosmo_fit,
    'psf',
    ab_minus_vega=ab_minus_vega,
)

photometry[
    [
        'mjd',
        'filter',
        'aperture_abmag',
        'aperture_abmag_err',
        'aperture_vega_mag',
        'psf_abmag',
        'psf_abmag_err',
        'psf_vega_mag',
        'sncosmo_model_abmag',
        'sncosmo_model_vega_mag',
    ]
][:8]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for prefix, marker, label in [
    ('aperture', 'o', 'Aperture photometry'),
    ('psf', 's', 'PSF photometry'),
]:
    mag_column = f'{prefix}_abmag'
    err_column = f'{prefix}_abmag_err'
    valid = np.isfinite(photometry[mag_column])
    ax.errorbar(
        photometry['mjd'][valid],
        photometry[mag_column][valid],
        yerr=photometry[err_column][valid],
        fmt=marker,
        label=label,
    )

ax.plot(dense_mjd, dense_abmag, color='0.2', label='Injected sncosmo model')
ax.axvline(model_peak_mjd, color='0.4', linestyle='--', linewidth=1, label='sncosmo peak')
ax.set_xlabel('MJD')
ax.set_ylabel('AB magnitude')
ax.set_title(f'{filter_name} model-calibrated magnitude light curve')
ax.invert_yaxis()
ax.grid(alpha=0.3)
ax.legend(fontsize=9)

if np.isfinite(ab_minus_vega):
    secax = ax.secondary_yaxis(
        'right',
        functions=(
            lambda abmag: abmag - ab_minus_vega,
            lambda vegamag: vegamag + ab_minus_vega,
        ),
    )
    secax.set_ylabel('Vega magnitude')

In [ ]:
fig, (ax_lc, ax_resid) = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(8, 7),
    sharex=True,
    gridspec_kw={'height_ratios': [3, 1]},
    constrained_layout=True,
)

aperture_dense_fit = aperture_sncosmo_fit['scale'] * dense_maggies + aperture_sncosmo_fit['offset']
psf_dense_fit = psf_sncosmo_fit['scale'] * dense_maggies + psf_sncosmo_fit['offset']

ax_lc.errorbar(
    photometry['mjd'],
    photometry['aperture_flux'],
    yerr=photometry['aperture_flux_err'],
    fmt='o',
    label='Aperture photometry',
)
ax_lc.plot(dense_mjd, aperture_dense_fit, color='C0', alpha=0.7, label='Scaled sncosmo model (aperture)')

ax_lc.errorbar(
    photometry['mjd'],
    photometry['psf_flux'],
    yerr=photometry['psf_flux_err'],
    fmt='s',
    label='PSF photometry',
)
ax_lc.plot(dense_mjd, psf_dense_fit, color='C1', alpha=0.7, label='Scaled sncosmo model (PSF)')

ax_lc.axvline(model_peak_mjd, color='0.3', linestyle='--', linewidth=1, label='sncosmo peak')
ax_lc.set_ylabel('Flux (image units)')
ax_lc.set_title(f'{filter_name} photometry compared with the input sncosmo model')
ax_lc.legend(fontsize=9)

ax_resid.axhline(0, color='0.4', linestyle='--', linewidth=1)
ax_resid.scatter(photometry['mjd'], photometry['aperture_sncosmo_residual'], label='Aperture residual')
ax_resid.scatter(photometry['mjd'], photometry['psf_sncosmo_residual'], label='PSF residual')
ax_resid.set_xlabel('MJD')
ax_resid.set_ylabel('Data - model')
ax_resid.legend(fontsize=9)

for ax in (ax_lc, ax_resid):
    ax.grid(alpha=0.3)

The fitted scale factors map the dimensionless `sncosmo` model fluxes into the image units measured from the simulated cutouts. The constant offsets are useful diagnostics: a large offset can indicate that residual host-galaxy light or sky background remains in the measurement. The residual panel is a quick check for time-dependent structure that is not captured by the input light-curve model.

## Injected-Truth Comparison

Because this is a simulation, we know the input `sncosmo` light-curve shape. The absolute image fluxes and `sncosmo` maggies live in different units, but after normalizing each curve by its own peak we can compare how well aperture and PSF photometry recover the injected time evolution.

In [ ]:
photometry['aperture_truth_residual_normalized'] = (
    photometry['aperture_flux_normalized'] - photometry['sncosmo_model_normalized']
)
photometry['psf_truth_residual_normalized'] = (
    photometry['psf_flux_normalized'] - photometry['sncosmo_model_normalized']
)

truth_summary = Table(rows=[
    {
        'method': 'Aperture',
        'rms_normalized_residual': np.sqrt(np.nanmean(photometry['aperture_truth_residual_normalized'] ** 2)),
        'median_normalized_residual': np.nanmedian(photometry['aperture_truth_residual_normalized']),
    },
    {
        'method': 'PSF',
        'rms_normalized_residual': np.sqrt(np.nanmean(photometry['psf_truth_residual_normalized'] ** 2)),
        'median_normalized_residual': np.nanmedian(photometry['psf_truth_residual_normalized']),
    },
])

truth_summary

In [ ]:
fig, (ax_lc, ax_resid) = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(8, 7),
    sharex=True,
    gridspec_kw={'height_ratios': [3, 1]},
    constrained_layout=True,
)

ax_lc.plot(dense_mjd, dense_maggies_normalized, color='0.2', label='Injected sncosmo truth')
ax_lc.scatter(photometry['mjd'], photometry['aperture_flux_normalized'], label='Aperture')
ax_lc.scatter(photometry['mjd'], photometry['psf_flux_normalized'], label='PSF')
ax_lc.set_ylabel('Flux / peak flux')
ax_lc.set_title(f'Normalized {filter_name} recovery compared with injected truth')
ax_lc.legend()

ax_resid.axhline(0, color='0.4', linestyle='--', linewidth=1)
ax_resid.scatter(
    photometry['mjd'],
    photometry['aperture_truth_residual_normalized'],
    label='Aperture - truth',
)
ax_resid.scatter(
    photometry['mjd'],
    photometry['psf_truth_residual_normalized'],
    label='PSF - truth',
)
ax_resid.set_xlabel('MJD')
ax_resid.set_ylabel('Residual')
ax_resid.legend()

for ax in (ax_lc, ax_resid):
    ax.grid(alpha=0.3)

## Difference-Image Photometry

A common time-domain strategy is to subtract a reference image before measuring the transient. For this simulated sequence, we choose the epoch where the input `sncosmo` model is faintest as a simple reference template. This is not a full image-subtraction pipeline, but it demonstrates how template subtraction changes the aperture and PSF light curves.

In [ ]:
template_index = int(np.nanargmin(photometry['sncosmo_model_maggies']))
template_image = images[template_index]
template_mjd = photometry['mjd'][template_index]
template_model_flux = photometry['sncosmo_model_maggies'][template_index]

difference_rows = []
for path, mjd, filt, image in zip(cutout_files, mjds, filters, images):
    difference_image = image - template_image
    aperture_result = measure_aperture_flux(
        difference_image,
        cutout_target_position,
        aperture_radius,
        annulus_inner_radius,
        annulus_outer_radius,
        correction=aperture_correction,
    )
    psf_result = measure_psf_flux(
        difference_image,
        cutout_target_position,
        full_detector_target_position,
        psf_grid,
        half_size=psf_fit_half_size,
    )
    difference_rows.append({
        'file': path.name,
        'mjd': mjd,
        'filter': filt,
        'difference_aperture_flux': aperture_result['aperture_flux'],
        'difference_aperture_flux_err': aperture_result['aperture_flux_err'],
        'difference_psf_flux': psf_result['psf_flux'],
        'difference_psf_flux_err': psf_result['psf_flux_err'],
    })

difference_table = Table(rows=difference_rows)
photometry['difference_aperture_flux'] = difference_table['difference_aperture_flux']
photometry['difference_aperture_flux_err'] = difference_table['difference_aperture_flux_err']
photometry['difference_psf_flux'] = difference_table['difference_psf_flux']
photometry['difference_psf_flux_err'] = difference_table['difference_psf_flux_err']
photometry['difference_model_maggies'] = photometry['sncosmo_model_maggies'] - template_model_flux
photometry['difference_aperture_snr'] = safe_ratio(
    photometry['difference_aperture_flux'],
    photometry['difference_aperture_flux_err'],
)
photometry['difference_psf_snr'] = safe_ratio(
    photometry['difference_psf_flux'],
    photometry['difference_psf_flux_err'],
)

print(f'Using MJD {template_mjd:.0f} as the difference-image reference epoch.')
difference_table[:5]

In [ ]:
difference_aperture_fit = fit_scaled_sncosmo_model(
    photometry['difference_aperture_flux'],
    photometry['difference_aperture_flux_err'],
    photometry['difference_model_maggies'],
)
difference_psf_fit = fit_scaled_sncosmo_model(
    photometry['difference_psf_flux'],
    photometry['difference_psf_flux_err'],
    photometry['difference_model_maggies'],
)

photometry['difference_aperture_sncosmo_fit_flux'] = difference_aperture_fit['fitted_flux']
photometry['difference_aperture_sncosmo_residual'] = difference_aperture_fit['residual']
photometry['difference_psf_sncosmo_fit_flux'] = difference_psf_fit['fitted_flux']
photometry['difference_psf_sncosmo_residual'] = difference_psf_fit['residual']

add_model_calibrated_magnitude_columns(
    photometry,
    'difference_aperture_flux',
    'difference_aperture_flux_err',
    difference_aperture_fit,
    'difference_aperture',
    ab_minus_vega=ab_minus_vega,
)
add_model_calibrated_magnitude_columns(
    photometry,
    'difference_psf_flux',
    'difference_psf_flux_err',
    difference_psf_fit,
    'difference_psf',
    ab_minus_vega=ab_minus_vega,
)

difference_sncosmo_summary = Table(rows=[
    {
        'method': 'Difference aperture',
        'template_mjd': template_mjd,
        'scale': difference_aperture_fit['scale'],
        'offset': difference_aperture_fit['offset'],
        'reduced_chi2': difference_aperture_fit['reduced_chi2'],
    },
    {
        'method': 'Difference PSF',
        'template_mjd': template_mjd,
        'scale': difference_psf_fit['scale'],
        'offset': difference_psf_fit['offset'],
        'reduced_chi2': difference_psf_fit['reduced_chi2'],
    },
])

difference_sncosmo_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.errorbar(
    photometry['mjd'],
    photometry['difference_aperture_flux'],
    yerr=photometry['difference_aperture_flux_err'],
    fmt='o-',
    label='Difference aperture',
)
ax.errorbar(
    photometry['mjd'],
    photometry['difference_psf_flux'],
    yerr=photometry['difference_psf_flux_err'],
    fmt='s-',
    label='Difference PSF',
)

ax.plot(
    photometry['mjd'],
    photometry['difference_aperture_sncosmo_fit_flux'],
    color='C0',
    alpha=0.7,
    label='Scaled model (difference aperture)',
)
ax.plot(
    photometry['mjd'],
    photometry['difference_psf_sncosmo_fit_flux'],
    color='C1',
    alpha=0.7,
    label='Scaled model (difference PSF)',
)

ax.axhline(0, color='0.4', linestyle='--', linewidth=1)
ax.axvline(template_mjd, color='0.5', linestyle=':', linewidth=1, label='Reference epoch')
ax.set_xlabel('MJD')
ax.set_ylabel('Difference flux (image units)')
ax.set_title(f'{filter_name} simple template-subtracted light curve')
ax.grid(alpha=0.3)
ax.legend(fontsize=9)

## Host-Galaxy and Background Diagnostics

The supernova was injected near a host galaxy, so residual host light can bias the measured transient flux. The diagnostics below look for correlations between photometric residuals and local background estimates. Strong trends can indicate that a method is still sensitive to host-galaxy structure or imperfect background removal.

In [ ]:
def finite_correlation(x, y):
    """Compute a Pearson correlation coefficient for finite values only."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if np.count_nonzero(mask) < 3:
        return np.nan
    return np.corrcoef(x[mask], y[mask])[0, 1]


photometry['local_background_difference'] = (
    photometry['background_per_pixel'] - photometry['psf_background']
)
photometry['absolute_truth_residual_difference'] = (
    np.abs(photometry['aperture_truth_residual_normalized'])
    - np.abs(photometry['psf_truth_residual_normalized'])
)

host_diagnostics = Table(rows=[
    {
        'diagnostic': 'Aperture residual vs aperture background',
        'correlation': finite_correlation(
            photometry['aperture_truth_residual_normalized'],
            photometry['background_per_pixel'],
        ),
    },
    {
        'diagnostic': 'PSF residual vs fitted PSF background',
        'correlation': finite_correlation(
            photometry['psf_truth_residual_normalized'],
            photometry['psf_background'],
        ),
    },
    {
        'diagnostic': 'Aperture / PSF flux ratio vs aperture background',
        'correlation': finite_correlation(
            photometry['flux_ratio'],
            photometry['background_per_pixel'],
        ),
    },
    {
        'diagnostic': 'Aperture background fraction vs truth residual difference',
        'correlation': finite_correlation(
            photometry['aperture_background_fraction'],
            photometry['absolute_truth_residual_difference'],
        ),
    },
])

host_diagnostics

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(11, 4), constrained_layout=True)

axes[0].scatter(
    photometry['background_per_pixel'],
    photometry['aperture_truth_residual_normalized'],
    label='Aperture',
)
axes[0].scatter(
    photometry['psf_background'],
    photometry['psf_truth_residual_normalized'],
    label='PSF',
)
axes[0].axhline(0, color='0.4', linestyle='--', linewidth=1)
axes[0].set_xlabel('Local background estimate')
axes[0].set_ylabel('Normalized residual from truth')
axes[0].set_title('Residuals vs. background')
axes[0].legend()

axes[1].scatter(photometry['mjd'], photometry['flux_ratio'])
axes[1].axhline(1, color='0.4', linestyle='--', linewidth=1)
axes[1].set_xlabel('MJD')
axes[1].set_ylabel('Aperture / PSF flux')
axes[1].set_title('Method ratio over time')

for ax in axes:
    ax.grid(alpha=0.3)

## Normalized sncosmo Fit for Peak Time

The previous `sncosmo` comparison held the simulation parameters fixed and only fit a flux scale plus constant offset. We can also ask whether the recovered normalized light curve prefers the same peak time, $t_0$, as the injected model.

Here we scan a small grid of possible `t0` values around the injected value. For each trial `t0`, we evaluate the SALT3-NIR model, normalize it, and fit a linear amplitude and offset to the normalized measured light curve.

In [ ]:
def fit_normalized_t0_grid(measured_normalized, measured_err, mjd, filter_name, t0_grid):
    """Fit normalized measured fluxes with a grid of sncosmo t0 values."""
    measured_normalized = np.asarray(measured_normalized, dtype=float)
    measured_err = np.asarray(measured_err, dtype=float)
    mjd = np.asarray(mjd, dtype=float)

    fit_model = sncosmo.Model(source='salt3-nir')
    fit_model.set(z=1.2, x1=1, c=1)
    fit_model.set_source_peakmag(20, 'f158', 'ab')

    rows = []
    for trial_t0 in t0_grid:
        fit_model.set(t0=trial_t0)
        trial_maggies = sncosmo_maggies_for_filter(filter_name, mjd, model=fit_model)
        trial_model = trial_maggies / np.nanmax(trial_maggies)
        fit = fit_scaled_sncosmo_model(measured_normalized, measured_err, trial_model)
        rows.append({
            't0': trial_t0,
            'scale': fit['scale'],
            'offset': fit['offset'],
            'reduced_chi2': fit['reduced_chi2'],
        })

    results = Table(rows=rows)
    best_index = int(np.nanargmin(results['reduced_chi2']))
    return results, results[best_index]


t0_grid = simulation_t0 + np.linspace(-10, 10, 201)
aperture_norm_err = safe_ratio(photometry['aperture_flux_err'], np.nanmax(photometry['aperture_flux']))
psf_norm_err = safe_ratio(photometry['psf_flux_err'], np.nanmax(photometry['psf_flux']))

aperture_t0_grid, aperture_best_t0 = fit_normalized_t0_grid(
    photometry['aperture_flux_normalized'],
    aperture_norm_err,
    photometry['mjd'],
    filter_name,
    t0_grid,
)
psf_t0_grid, psf_best_t0 = fit_normalized_t0_grid(
    photometry['psf_flux_normalized'],
    psf_norm_err,
    photometry['mjd'],
    filter_name,
    t0_grid,
)

t0_fit_summary = Table(rows=[
    {
        'method': 'Aperture',
        'best_t0': aperture_best_t0['t0'],
        't0_offset_days': aperture_best_t0['t0'] - simulation_t0,
        'scale': aperture_best_t0['scale'],
        'offset': aperture_best_t0['offset'],
        'reduced_chi2': aperture_best_t0['reduced_chi2'],
    },
    {
        'method': 'PSF',
        'best_t0': psf_best_t0['t0'],
        't0_offset_days': psf_best_t0['t0'] - simulation_t0,
        'scale': psf_best_t0['scale'],
        'offset': psf_best_t0['offset'],
        'reduced_chi2': psf_best_t0['reduced_chi2'],
    },
])

t0_fit_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(aperture_t0_grid['t0'] - simulation_t0, aperture_t0_grid['reduced_chi2'], label='Aperture')
ax.plot(psf_t0_grid['t0'] - simulation_t0, psf_t0_grid['reduced_chi2'], label='PSF')
ax.axvline(0, color='0.4', linestyle='--', linewidth=1, label='Injected t0')

ax.set_xlabel('Trial t0 - injected t0 (days)')
ax.set_ylabel('Reduced chi-square')
ax.set_title(f'{filter_name} normalized t0 scan')
ax.grid(alpha=0.3)
ax.legend()

## Multi-Filter Photometry

The default simulation notebook writes F129 cutouts, and includes commented lines for F184. The functions above are filter-aware, so if additional files matching `roman_sn1a_*_*.asdf` are present, the cell below measures each filter and builds a combined photometry table.

For filters beyond the primary walkthrough filter, the notebook computes a matching aperture correction and STPSF grid before measuring aperture and PSF fluxes.

In [ ]:
def get_aperture_correction(filter_name):
    """Return a cached aperture correction for one filter."""
    if filter_name not in aperture_correction_cache:
        aperture_correction_cache[filter_name] = aperture_correction_from_stpsf(
            filter_name=filter_name,
            detector_name=detector_name,
            detector_position=full_detector_target_position,
            radius_pix=aperture_radius,
        )
    return aperture_correction_cache[filter_name]


def get_psf_grid(filter_name):
    """Return a cached STPSF grid for one filter."""
    if filter_name not in psf_grid_cache:
        psf_grid_cache[filter_name] = make_psf_grid(
            filter_name,
            detector_name,
            full_detector_target_position,
        )
    return psf_grid_cache[filter_name]


def measure_cutout_file(path):
    """Measure aperture and PSF photometry for one ASDF cutout file."""
    filt = extract_filter(path)
    image = read_cutout(path)
    correction = get_aperture_correction(filt)
    grid = get_psf_grid(filt)

    aperture_result = measure_aperture_flux(
        image,
        cutout_target_position,
        aperture_radius,
        annulus_inner_radius,
        annulus_outer_radius,
        correction=correction,
    )
    psf_result = measure_psf_flux(
        image,
        cutout_target_position,
        full_detector_target_position,
        grid,
        half_size=psf_fit_half_size,
    )

    mjd = extract_mjd(path)
    model_abmag = sncosmo_model.bandmag(filt.lower(), 'ab', mjd)
    model_maggies = 10 ** (-0.4 * model_abmag)

    return {
        'file': path.name,
        'mjd': mjd,
        'filter': filt,
        'aperture_flux': aperture_result['aperture_flux'],
        'aperture_flux_err': aperture_result['aperture_flux_err'],
        'psf_flux': psf_result['psf_flux'],
        'psf_flux_err': psf_result['psf_flux_err'],
        'psf_background': psf_result['psf_background'],
        'psf_background_x_slope': psf_result['psf_background_x_slope'],
        'psf_background_y_slope': psf_result['psf_background_y_slope'],
        'psf_x': psf_result['psf_x'],
        'psf_y': psf_result['psf_y'],
        'psf_residual_rms': psf_result['psf_residual_rms'],
        'psf_max_abs_residual_fraction': psf_result['psf_max_abs_residual_fraction'],
        'aperture_snr': safe_ratio(
            aperture_result['aperture_flux'],
            aperture_result['aperture_flux_err'],
        ),
        'psf_snr': safe_ratio(
            psf_result['psf_flux'],
            psf_result['psf_flux_err'],
        ),
        'sncosmo_model_abmag': model_abmag,
        'sncosmo_model_maggies': model_maggies,
    }


multi_filter_photometry = Table(rows=[
    measure_cutout_file(path)
    for path in all_cutout_files
])

for column in ('aperture_flux', 'psf_flux', 'sncosmo_model_maggies'):
    multi_filter_photometry[f'{column}_normalized'] = np.nan

for filt in sorted(set(multi_filter_photometry['filter'])):
    filt_mask = multi_filter_photometry['filter'] == filt
    for column in ('aperture_flux', 'psf_flux', 'sncosmo_model_maggies'):
        peak_value = np.nanmax(multi_filter_photometry[column][filt_mask])
        multi_filter_photometry[f'{column}_normalized'][filt_mask] = (
            multi_filter_photometry[column][filt_mask] / peak_value
        )

    filt_ab_minus_vega = get_ab_minus_vega_offset(filt, multi_filter_photometry['mjd'][filt_mask])
    ensure_float_column(multi_filter_photometry, 'sncosmo_model_vega_mag')
    if np.isfinite(filt_ab_minus_vega):
        multi_filter_photometry['sncosmo_model_vega_mag'][filt_mask] = (
            multi_filter_photometry['sncosmo_model_abmag'][filt_mask] - filt_ab_minus_vega
        )

    filt_aperture_fit = fit_scaled_sncosmo_model(
        multi_filter_photometry['aperture_flux'][filt_mask],
        multi_filter_photometry['aperture_flux_err'][filt_mask],
        multi_filter_photometry['sncosmo_model_maggies'][filt_mask],
    )
    filt_psf_fit = fit_scaled_sncosmo_model(
        multi_filter_photometry['psf_flux'][filt_mask],
        multi_filter_photometry['psf_flux_err'][filt_mask],
        multi_filter_photometry['sncosmo_model_maggies'][filt_mask],
    )
    add_model_calibrated_magnitude_columns(
        multi_filter_photometry,
        'aperture_flux',
        'aperture_flux_err',
        filt_aperture_fit,
        'aperture',
        ab_minus_vega=filt_ab_minus_vega,
        mask=filt_mask,
    )
    add_model_calibrated_magnitude_columns(
        multi_filter_photometry,
        'psf_flux',
        'psf_flux_err',
        filt_psf_fit,
        'psf',
        ab_minus_vega=filt_ab_minus_vega,
        mask=filt_mask,
    )

multi_filter_photometry[:8]

## Save the Photometry Tables

The combined single-filter table and the optional multi-filter table can be written to ECSV files for follow-up modeling or comparison with the input `sncosmo` model from the simulation notebook. These tables include image-flux measurements, model-calibrated AB magnitudes, and Vega magnitudes when the local Vega magnitude system is available.

In [ ]:
output_table = Path('time_domain_analysis_photometry.ecsv')
photometry.write(output_table, format='ascii.ecsv', overwrite=True)
print(f'Wrote {output_table}')

multi_filter_output_table = Path('time_domain_analysis_multifilter_photometry.ecsv')
multi_filter_photometry.write(multi_filter_output_table, format='ascii.ecsv', overwrite=True)
print(f'Wrote {multi_filter_output_table}')

## Animate the Supernova and Light Curve

As a final visual check, we can combine the image sequence with the measured light curve. The left panel shows the evolving cutout, while the right panel advances through the PSF photometry points from the same epochs. This links each image frame directly to its position on the recovered light curve.

When the host-containing simulation GIF is available, this cell uses those frames for the image panel so the full galaxy scene from the simulation notebook is preserved. The light curve still comes from the photometry measured in this analysis notebook.

In [ ]:
def resolve_cutout_path(file_name):
    """Find a cutout file in the current directory or the demo-data directory."""
    path = Path(str(file_name))
    if path.exists():
        return path

    demo_path = Path('demo_cutouts') / path.name
    if demo_path.exists():
        return demo_path

    raise FileNotFoundError(f'Could not find cutout file: {file_name}')


def find_dark_image_crop(frame):
    """Crop the black image panel out of a Matplotlib-rendered GIF frame."""
    dark_pixels = np.mean(frame, axis=2) < 35
    row_fraction = dark_pixels.mean(axis=1)
    column_fraction = dark_pixels.mean(axis=0)
    rows = np.where(row_fraction > 0.5)[0]
    columns = np.where(column_fraction > 0.5)[0]

    if len(rows) == 0 or len(columns) == 0:
        return (0, frame.shape[0], 0, frame.shape[1])

    return (rows[0], rows[-1] + 1, columns[0], columns[-1] + 1)


def load_simulation_gif_frames(number_of_frames):
    """Load host-galaxy frames from the simulation GIF when it is available."""
    candidate_gifs = [
        Path('simulation_sn1a.gif'),
        Path('../time_domain_simulations/sn1a.gif'),
    ]
    existing_gifs = [path for path in candidate_gifs if path.exists()]
    if len(existing_gifs) == 0:
        return None, None

    source_gif = existing_gifs[0]
    with PILImage.open(source_gif) as gif:
        frames = [
            np.asarray(frame.convert('RGB'), dtype=np.uint8)
            for frame in ImageSequence.Iterator(gif)
        ]

    if len(frames) != number_of_frames:
        frame_indices = np.linspace(0, len(frames) - 1, number_of_frames)
        frame_indices = np.round(frame_indices).astype(int)
        frames = [frames[index] for index in frame_indices]

    row_start, row_stop, column_start, column_stop = find_dark_image_crop(frames[0])
    frames = [
        frame[row_start:row_stop, column_start:column_stop]
        for frame in frames
    ]

    return frames, source_gif


animation_table = photometry.copy() if 'photometry' in globals() else Table.read(
    'time_domain_analysis_photometry.ecsv'
)
animation_table.sort('mjd')

animation_filter = filter_name if 'filter_name' in globals() else str(animation_table['filter'][0])
if 'filter' in animation_table.colnames:
    animation_table = animation_table[animation_table['filter'] == animation_filter]

cutout_paths = [resolve_cutout_path(file_name) for file_name in animation_table['file']]
mjd_values = np.asarray(animation_table['mjd'], dtype=float)
psf_curve = np.asarray(animation_table['psf_flux_normalized'], dtype=float)

image_frames, image_source = load_simulation_gif_frames(len(animation_table))
using_gif_frames = image_frames is not None

if not using_gif_frames:
    image_frames = []
    for path in cutout_paths:
        with asdf.open(path) as af:
            image_frames.append(np.asarray(af['roman']['data'], dtype=float))
    image_source = 'ASDF cutouts'

figure, (image_ax, light_curve_ax) = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(11, 4.5),
    gridspec_kw={'width_ratios': [1, 1.25]},
    constrained_layout=True,
)

if using_gif_frames:
    image_artist = image_ax.imshow(image_frames[0], animated=True)
    image_ax.set_xticks([])
    image_ax.set_yticks([])
else:
    image_norm = simple_norm(image_frames[0], 'asinh', vmin=0.5, vmax=3)
    image_artist = image_ax.imshow(
        image_frames[0],
        origin='lower',
        cmap='afmhot',
        norm=image_norm,
        animated=True,
    )

image_ax.set_title(f'{animation_filter} cutout: MJD {mjd_values[0]:.0f}')

optional_y_values = [psf_curve]
if 'sncosmo_model_normalized' in animation_table.colnames:
    model_curve = np.asarray(animation_table['sncosmo_model_normalized'], dtype=float)
    optional_y_values.append(model_curve)
    light_curve_ax.plot(
        mjd_values,
        model_curve,
        color='0.35',
        linewidth=1.6,
        label='sncosmo truth',
    )

if 'aperture_flux_normalized' in animation_table.colnames:
    aperture_curve = np.asarray(animation_table['aperture_flux_normalized'], dtype=float)
    optional_y_values.append(aperture_curve)
    light_curve_ax.scatter(
        mjd_values,
        aperture_curve,
        facecolors='none',
        edgecolors='tab:blue',
        s=34,
        alpha=0.45,
        label='Aperture',
    )

light_curve_ax.scatter(
    mjd_values,
    psf_curve,
    color='tab:orange',
    s=32,
    alpha=0.35,
    label='PSF',
)
progress_line, = light_curve_ax.plot([], [], color='tab:orange', linewidth=2.0)
progress_points = light_curve_ax.scatter([], [], color='tab:orange', s=36, zorder=4)
current_point = light_curve_ax.scatter(
    [mjd_values[0]],
    [psf_curve[0]],
    color='crimson',
    edgecolor='white',
    linewidth=0.8,
    s=95,
    zorder=5,
)
current_epoch = light_curve_ax.axvline(
    mjd_values[0],
    color='crimson',
    alpha=0.35,
    linewidth=1.2,
)
epoch_label = light_curve_ax.text(
    0.03,
    0.95,
    '',
    transform=light_curve_ax.transAxes,
    ha='left',
    va='top',
)

finite_y = np.concatenate([
    values[np.isfinite(values)]
    for values in optional_y_values
])
y_min = min(0.0, float(np.nanmin(finite_y)) - 0.05)
y_max = max(1.05, float(np.nanmax(finite_y)) + 0.08)

light_curve_ax.set_xlim(np.nanmin(mjd_values) - 2, np.nanmax(mjd_values) + 2)
light_curve_ax.set_ylim(y_min, y_max)
light_curve_ax.set_xlabel('MJD')
light_curve_ax.set_ylabel('Flux / peak flux')
light_curve_ax.set_title('Recovered light curve')
light_curve_ax.grid(alpha=0.3)
light_curve_ax.legend(loc='upper right', frameon=False)


def update_animation(frame_index):
    image_artist.set_data(image_frames[frame_index])
    image_ax.set_title(
        f'{animation_filter} cutout: MJD {mjd_values[frame_index]:.0f}'
    )

    progress_line.set_data(
        mjd_values[: frame_index + 1],
        psf_curve[: frame_index + 1],
    )
    progress_points.set_offsets(
        np.column_stack([
            mjd_values[: frame_index + 1],
            psf_curve[: frame_index + 1],
        ])
    )
    current_point.set_offsets([[mjd_values[frame_index], psf_curve[frame_index]]])
    current_epoch.set_xdata([mjd_values[frame_index], mjd_values[frame_index]])
    epoch_label.set_text(
        f'MJD {mjd_values[frame_index]:.0f}\n'
        f'PSF flux / peak = {psf_curve[frame_index]:.2f}'
    )

    return (
        image_artist,
        progress_line,
        progress_points,
        current_point,
        current_epoch,
        epoch_label,
    )


light_curve_animation = animation.FuncAnimation(
    figure,
    update_animation,
    frames=len(image_frames),
    interval=350,
    blit=False,
    repeat_delay=1000,
)
output_animation = Path('sn1a_lightcurve.gif')
light_curve_animation.save(
    output_animation,
    writer=animation.PillowWriter(fps=4),
)
plt.close(figure)

print(f'Image frames: {image_source}')
print(f'Wrote {output_animation}')
display(NotebookImage(filename=str(output_animation)))

## Additional Resources

- [Time-domain simulation notebook](../time_domain_simulations/time_domain_simulations.ipynb)
- [Aperture Photometry notebook](../aperture_photometry/aperture_photometry.ipynb)
- [STPSF tutorial](../stpsf/stpsf.ipynb)
- [Photutils aperture photometry documentation](https://photutils.readthedocs.io/en/latest/user_guide/aperture.html)
- [Photutils PSF photometry documentation](https://photutils.readthedocs.io/en/latest/user_guide/psf.html)
- [sncosmo documentation](https://sncosmo.readthedocs.io/en/stable/)
- [Roman User Documentation -- RDox](https://roman-docs.stsci.edu/)

***

## About This Notebook

**Author:** Melissa Shahbandeh\
**Updated On:** 2026-07-15

<table width="100%" style="border:none; border-collapse:collapse;">
  <tr style="border:none;">
    <td style="border:none; width:180px; white-space:nowrap;">
       <a href="#top" style="text-decoration:none; color:#0066cc;"> Top of page</a>
    </td>
    <td style="border:none; text-align:center;">
        <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/roman_logo.png" alt="roman_logo" width="50px">
    </td>
    <td style="border:none; text-align:right;">
       <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/stsci_logo2.png" width="90">
    </td>
  </tr>
</table>